# ClearScape Vantage Workload Telemetry Demo

Notebook-first walkthrough for deriving workloads from Vantage-native telemetry (DBQL + ResUsage), generating controlled activity, and drilling back to raw telemetry IDs (`QueryID`, `StepID`, sample time, node).

## 1) Setup

This section configures connection behavior, runtime parameters, and capability checks before running any demo logic.

In [ ]:
from __future__ import annotations

import os
from datetime import datetime
from typing import Iterable, Optional

import pandas as pd
import numpy as np

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

In [ ]:
CONFIG = {
    "profile": os.getenv("VANTAGE_PROFILE", "default"),
    "host": os.getenv("VANTAGE_HOST"),
    "user": os.getenv("VANTAGE_USER"),
    "password": os.getenv("VANTAGE_PASSWORD"),
    "window_minutes": 120,
    "bucket_minutes": 5,
    "demo_schema": os.getenv("DEMO_SCHEMA", "demo_workload_v1"),
    "run_label": os.getenv("RUN_LABEL", datetime.utcnow().strftime("run_%Y%m%d_%H%M%S")),
    "generate_demo_workload": True,
}
CONFIG

In [ ]:
conn = None

try:
    from teradataml import create_context, get_context
    if CONFIG["host"] and CONFIG["user"] and CONFIG["password"]:
        create_context(host=CONFIG["host"], username=CONFIG["user"], password=CONFIG["password"])
    conn = get_context()
    print("Connected via teradataml context")
except Exception as e_teradataml:
    print(f"teradataml unavailable or not configured: {e_teradataml}")
    try:
        import teradatasql
        if not (CONFIG["host"] and CONFIG["user"] and CONFIG["password"]):
            raise RuntimeError("Set VANTAGE_HOST/VANTAGE_USER/VANTAGE_PASSWORD when profile context is unavailable.")
        conn = teradatasql.connect(host=CONFIG["host"], user=CONFIG["user"], password=CONFIG["password"])
        print("Connected via teradatasql")
    except Exception as e_td:
        raise RuntimeError("Unable to establish Vantage connection. Configure ClearScape profile or env vars.") from e_td

def read_sql(sql: str, params: Optional[dict] = None) -> pd.DataFrame:
    params = params or {}
    try:
        return pd.read_sql(sql, conn, params=params)
    except Exception:
        return pd.read_sql(sql, conn)

def execute_sql(sql: str) -> None:
    cur = conn.cursor()
    cur.execute(sql)
    cur.close()

In [ ]:
SURFACES = {
    "dbql_query": ["DBC.DBQLogTblV", "DBC.QryLogV"],
    "dbql_step": ["DBC.DBQLStepTblV", "DBC.DBQLStepTbl"],
    "resusage": ["DBC.ResUsageSpmaV", "DBC.ResUsageSpsV"],
}

def first_accessible_object(candidates: Iterable[str]):
    for obj in candidates:
        try:
            read_sql(f"SELECT TOP 1 * FROM {obj}")
            return obj
        except Exception:
            continue
    return None

telemetry_objects = {k: first_accessible_object(v) for k, v in SURFACES.items()}
telemetry_objects_df = pd.DataFrame([
    {"family": k, "selected_object": v, "available": v is not None}
    for k, v in telemetry_objects.items()
])
display(telemetry_objects_df)

if telemetry_objects["dbql_query"] is None:
    raise RuntimeError("No DBQL query-level object is accessible. Cannot proceed with demo.")

## 2) Controlled workload generation

Creates a small schema and deterministic query mix to produce distinct derived workload signatures (`etl`, `bi`, `ad_hoc`, `heavy_skew`).

In [ ]:
demo_schema = CONFIG["demo_schema"]
run_label = CONFIG["run_label"]

if CONFIG["generate_demo_workload"]:
    try:
        execute_sql(f"CREATE DATABASE {demo_schema} AS PERM = 200000000;")
    except Exception:
        pass

    statements = [
        f"""
        CREATE MULTISET TABLE {demo_schema}.fact_sales AS (
            SELECT ROW_NUMBER() OVER (ORDER BY Calendar_Date) AS sale_id,
                   Calendar_Date AS sale_dt,
                   (EXTRACT(MONTH FROM Calendar_Date) * 17) MOD 1000 AS customer_id,
                   (EXTRACT(DAY FROM Calendar_Date) * 31) MOD 500 AS product_id,
                   (EXTRACT(DAY FROM Calendar_Date) * 9) MOD 7 AS region_id,
                   (EXTRACT(DAY FROM Calendar_Date) * 13) MOD 10000 AS amount
            FROM SYS_CALENDAR.CALENDAR
            SAMPLE 200000
        ) WITH DATA PRIMARY INDEX (customer_id);
        """,
        f"""
        CREATE MULTISET TABLE {demo_schema}.dim_customer AS (
            SELECT DISTINCT customer_id,
                'seg_' || TRIM((customer_id MOD 9)(FORMAT '9')) AS segment,
                'acct_' || TRIM((customer_id MOD 97)(FORMAT '99')) AS account_tier
            FROM {demo_schema}.fact_sales
        ) WITH DATA PRIMARY INDEX (customer_id);
        """,
        f"""
        CREATE MULTISET TABLE {demo_schema}.dim_product AS (
            SELECT DISTINCT product_id,
                'cat_' || TRIM((product_id MOD 11)(FORMAT '99')) AS category
            FROM {demo_schema}.fact_sales
        ) WITH DATA PRIMARY INDEX (product_id);
        """,
    ]
    for stmt in statements:
        try:
            execute_sql(stmt)
        except Exception:
            pass

    workload_sql = [
        f"SELECT COUNT(*) AS c FROM {demo_schema}.fact_sales WHERE sale_id < 1000;",
        f"SELECT customer_id, SUM(amount) FROM {demo_schema}.fact_sales GROUP BY 1 QUALIFY ROW_NUMBER() OVER (ORDER BY SUM(amount) DESC) <= 10;",
        f"""
        SELECT sale_dt, region_id, SUM(amount) AS total_amt, COUNT(*) AS txn_count
        FROM {demo_schema}.fact_sales
        WHERE sale_dt >= CURRENT_DATE - INTERVAL '60' DAY
        GROUP BY 1,2 ORDER BY 1,2;
        """,
        f"""
        CREATE VOLATILE TABLE vt_etl_{run_label[-6:]} AS (
            SELECT fs.customer_id, fs.product_id, SUM(fs.amount) AS amt, COUNT(*) AS txn
            FROM {demo_schema}.fact_sales fs
            JOIN {demo_schema}.dim_customer dc ON fs.customer_id = dc.customer_id
            JOIN {demo_schema}.dim_product dp ON fs.product_id = dp.product_id
            GROUP BY 1,2
        ) WITH DATA ON COMMIT PRESERVE ROWS;
        """,
        f"""
        SELECT COUNT(*)
        FROM {demo_schema}.fact_sales a
        JOIN {demo_schema}.fact_sales b ON a.customer_id = b.customer_id
        WHERE a.sale_id < 50000 AND b.sale_id < 50000;
        """,
    ]
    for q in workload_sql:
        try:
            execute_sql(q)
        except Exception as ex:
            print("Workload query failed (continuing):", ex)

{"demo_schema": demo_schema, "run_label": run_label}

## 3) Telemetry extraction (DBQL + Step + ResUsage)

Derive workload buckets from DBQL fields and keep raw telemetry IDs as first-class columns.

In [ ]:
window_minutes = CONFIG["window_minutes"]
bucket_minutes = CONFIG["bucket_minutes"]
query_obj = telemetry_objects["dbql_query"]
step_obj = telemetry_objects["dbql_step"]
res_obj = telemetry_objects["resusage"]

base_query_sql = f"""
WITH q AS (
    SELECT QueryID, UserName, SessionID, StartTime, FirstRespTime,
           COALESCE(TotalIOCount, ReqIOKB, 0) AS io_metric,
           COALESCE(AMPCPUTime, CPUSeconds, 0) AS cpu_metric,
           COALESCE(TotalFirstRespTime, 0) AS elapsed_metric,
           QueryText
    FROM {query_obj}
    WHERE StartTime >= CURRENT_TIMESTAMP - INTERVAL '{window_minutes}' MINUTE
), classified AS (
    SELECT QueryID, UserName, SessionID, StartTime, FirstRespTime,
           io_metric, cpu_metric, elapsed_metric, QueryText,
           CASE
             WHEN io_metric > 100000 OR elapsed_metric > 300 THEN 'etl'
             WHEN elapsed_metric BETWEEN 30 AND 300 THEN 'bi'
             WHEN elapsed_metric < 30 AND io_metric < 20000 THEN 'ad_hoc'
             ELSE 'ad_hoc'
           END AS derived_workload
    FROM q
)
SELECT * FROM classified;
"""

dbql_queries = read_sql(base_query_sql)
dbql_queries["bucket_ts"] = pd.to_datetime(dbql_queries["StartTime"]).dt.floor(f"{bucket_minutes}min")

if step_obj:
    step_sql = f"""
    SELECT QueryID, StepID,
           COALESCE(MAXAMPIO, MaxAmpIO, 0) AS max_amp_io,
           COALESCE(MINAMPIO, MinAmpIO, 0) AS min_amp_io,
           COALESCE(AMPCPUTime, 0) AS step_cpu,
           COALESCE(AMPIO, 0) AS step_io
    FROM {step_obj}
    WHERE CollectTimeStamp >= CURRENT_TIMESTAMP - INTERVAL '{window_minutes}' MINUTE;
    """
    try:
        step_df = read_sql(step_sql)
        step_df["skew_ratio"] = np.where(step_df["min_amp_io"] > 0, step_df["max_amp_io"] / step_df["min_amp_io"], np.nan)
    except Exception as ex:
        print("Step telemetry unavailable:", ex)
        step_df = pd.DataFrame()
else:
    print("No DBQL step object available; skew analysis skipped.")
    step_df = pd.DataFrame()

if res_obj:
    res_sql = f"""
    SELECT TheTime AS sample_time, NodeID,
           COALESCE(DiskReads, 0) AS disk_reads,
           COALESCE(DiskWrites, 0) AS disk_writes,
           COALESCE(CPUUServ, 0) AS cpu_user
    FROM {res_obj}
    WHERE TheTime >= CURRENT_TIMESTAMP - INTERVAL '{window_minutes}' MINUTE;
    """
    try:
        res_df = read_sql(res_sql)
        res_df["bucket_ts"] = pd.to_datetime(res_df["sample_time"]).dt.floor(f"{bucket_minutes}min")
    except Exception as ex:
        print("ResUsage unavailable:", ex)
        res_df = pd.DataFrame()
else:
    print("No ResUsage object available; system correlation skipped.")
    res_df = pd.DataFrame()

print("Rows:", {"dbql": len(dbql_queries), "step": len(step_df), "resusage": len(res_df)})

## 4) Analysis walkthrough

Workload pressure, IO outlier detection, and skew drilldown with raw telemetry IDs.

In [ ]:
workload_pressure = (
    dbql_queries.groupby(["bucket_ts", "derived_workload"], as_index=False)
    .agg(query_count=("QueryID", "nunique"), cpu_total=("cpu_metric", "sum"), io_total=("io_metric", "sum"), elapsed_total=("elapsed_metric", "sum"))
    .sort_values(["bucket_ts", "derived_workload"])
)
display(workload_pressure.head(50))

top_offenders = (
    dbql_queries.sort_values(["io_metric", "cpu_metric", "elapsed_metric"], ascending=False)
    [["QueryID", "derived_workload", "UserName", "SessionID", "StartTime", "elapsed_metric", "cpu_metric", "io_metric"]]
    .head(30)
)
display(top_offenders)

In [ ]:
io_by_bucket = workload_pressure.pivot_table(index="bucket_ts", columns="derived_workload", values="io_total", fill_value=0).sort_index()
io_long = io_by_bucket.reset_index().melt(id_vars=["bucket_ts"], var_name="derived_workload", value_name="io_total")
io_long["baseline_mean"] = io_long.groupby("derived_workload")["io_total"].transform("mean")
io_long["baseline_std"] = io_long.groupby("derived_workload")["io_total"].transform("std").replace(0, np.nan)
io_long["io_zscore"] = (io_long["io_total"] - io_long["baseline_mean"]) / io_long["baseline_std"]
io_outliers = io_long[io_long["io_zscore"] >= 2].sort_values("io_zscore", ascending=False)
display(io_outliers.head(20))

if not res_df.empty:
    res_overlay = res_df.groupby(["bucket_ts", "NodeID"], as_index=False).agg(
        disk_reads=("disk_reads", "sum"), disk_writes=("disk_writes", "sum"), cpu_user=("cpu_user", "mean")
    )
    display(res_overlay.head(50))
    if plt is not None:
        fig, ax = plt.subplots(figsize=(10, 4))
        workload_series = io_long.groupby("bucket_ts", as_index=False)["io_total"].sum()
        system_series = res_overlay.groupby("bucket_ts", as_index=False)["disk_reads"].sum()
        ax.plot(workload_series["bucket_ts"], workload_series["io_total"], label="DBQL IO total")
        ax.plot(system_series["bucket_ts"], system_series["disk_reads"], label="ResUsage DiskReads")
        ax.set_title("Workload IO vs ResUsage Disk Reads")
        ax.legend()
        plt.show()
else:
    print("ResUsage unavailable: DBQL analysis retained; system correlation unavailable.")

In [ ]:
if step_df.empty:
    print("No step telemetry available: skipping skew section with explicit note.")
else:
    skew_offenders = (
        step_df.sort_values(["skew_ratio", "step_io", "step_cpu"], ascending=False)
        [["QueryID", "StepID", "skew_ratio", "step_io", "step_cpu"]]
        .head(30)
    )
    display(skew_offenders)
    skew_with_query = skew_offenders.merge(
        dbql_queries[["QueryID", "derived_workload", "UserName", "SessionID", "StartTime", "QueryText"]],
        on="QueryID",
        how="left",
    )
    display(skew_with_query.head(30))

In [ ]:
checks = {}
checks["bucket_presence"] = sorted(dbql_queries["derived_workload"].dropna().unique().tolist())
checks["offenders_have_queryid"] = "QueryID" in top_offenders.columns
checks["skew_has_queryid_stepid"] = step_df.empty or ({"QueryID", "StepID"}.issubset(set(step_df.columns)))
checks["resusage_has_time_node"] = res_df.empty or ({"sample_time", "NodeID"}.issubset(set(res_df.columns)))

bad_actor_candidates = dbql_queries[
    dbql_queries["QueryText"].str.contains("JOIN", case=False, na=False)
].sort_values(["io_metric", "elapsed_metric"], ascending=False).head(5)
checks["bad_actor_traceable"] = not bad_actor_candidates.empty

display(pd.DataFrame([checks]))
display(bad_actor_candidates[["QueryID", "derived_workload", "elapsed_metric", "io_metric", "QueryText"]])

### Partial telemetry behavior
- No DBQL query object: notebook raises and stops.
- No step detail: skew analysis is skipped with an explicit note.
- No ResUsage access: DBQL workload/query analysis runs; system overlay is marked unavailable.